# Consumer: Image Inference

This notebook shows how an external consumer can:
1. Call `/embed/image` to map a raw image into the **exact** feature space used during training
2. Inspect the feature vector — visual tags (boolean) and GLCM texture features (continuous)
3. Call `/predict/image` to get a model prediction

The guarantee:
- **GLCM features** are computed with the same deterministic algorithm as training (no LLM)
- **Visual tag features** are scored by an LLM constrained to the exact tag vocabulary
  selected during training (`tag_vocabulary.json`) — no new tags can be introduced
- The output vector is always aligned to `feature_names.txt` and passed through
  the same `scaler.pkl` fitted on the training split

**Prerequisites:** The inference API must be running. Set `INFERENCE_URL` below.

## 1. Configuration

In [ ]:
import os

# When running via docker-compose, the service is on host port 8890.
# When deployed standalone on Kubeflow, use the cluster-internal URL.
INFERENCE_URL = os.getenv("INFERENCE_API_URL", "http://localhost:8890")

print(f"Inference API: {INFERENCE_URL}")

## 2. Health check

In [ ]:
import requests

resp = requests.get(f"{INFERENCE_URL}/health")
resp.raise_for_status()
health = resp.json()

print(f"Status         : {health['status']}")
print(f"Extractor ready: {health['extractor_ready']}")
print(f"Model ready    : {health['model_ready']}")
print(f"Data type      : {health['data_type']}")
print(f"N features     : {health['n_features']}")

## 3. Inspect the training feature space

The `/features` endpoint returns two groups of feature names:
- `tags` — boolean features scored by the LLM during training (and constrained at inference)
- `image_feature_names` — deterministic GLCM and statistical features

Together they form the full `feature_names` list in the order the model expects.

In [ ]:
resp = requests.get(f"{INFERENCE_URL}/features")
resp.raise_for_status()
features_meta = resp.json()

print(f"Data type      : {features_meta['data_type']}")
print(f"Total features : {features_meta['n_features']}")
print(f"  Visual tags  : {len(features_meta['tags'])}")
print(f"  Image feats  : {len(features_meta['image_feature_names'])}")
print()
print("Visual tag vocabulary (constrained at inference):")
for t in features_meta['tags']:
    print(f"  {t}")
print()
print("Image feature names (deterministic GLCM + stats):")
for t in features_meta['image_feature_names']:
    print(f"  {t}")

## 4. Load a test image

Set `IMAGE_PATH` to a local image file, or use the upload widget below.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Image as IPImage

IMAGE_PATH = None  # set to a local path to skip the uploader
_image_bytes = None

if IMAGE_PATH is None:
    uploader = widgets.FileUpload(accept="image/*", multiple=False)
    status   = widgets.HTML("<i style='color:grey'>No image uploaded yet.</i>")

    def _on_upload(change):
        global _image_bytes
        entries = change["new"]
        entries = entries if isinstance(entries, (list, tuple)) else list(entries.values())
        if entries:
            raw = entries[0]["content"]
            _image_bytes = bytes(raw) if not isinstance(raw, bytes) else raw
            status.value = f"<b style='color:green'>Loaded {len(_image_bytes):,} bytes</b>"

    uploader.observe(_on_upload, names="value")
    display(widgets.VBox([uploader, status]))
else:
    with open(IMAGE_PATH, "rb") as fh:
        _image_bytes = fh.read()
    print(f"Loaded {len(_image_bytes):,} bytes from {IMAGE_PATH}")

In [ ]:
# Preview the image
assert _image_bytes is not None, "Upload an image in the cell above first."
display(IPImage(data=_image_bytes, width=300))

## 5. Embed the image

In [ ]:
resp = requests.post(
    f"{INFERENCE_URL}/embed/image",
    files={"image": ("image.jpg", _image_bytes, "image/jpeg")},
)
resp.raise_for_status()
embed = resp.json()

print(f"N features returned: {embed['n_features']}")
print(f"Data type          : {embed['data_type']}")

## 6. Verify alignment with training features

In [ ]:
assert embed['feature_names'] == features_meta['feature_names'], (
    "Feature name mismatch between /embed/image and /features — this should never happen."
)
print("✓ Feature names are identical to the training feature space.")

## 7. Visualise the feature vector

Tag features (boolean) and GLCM/statistical features (continuous) are shown
in separate panels so the different scales don't collapse the view.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

all_names  = embed['feature_names']
all_values = embed['features']
feat_dict  = dict(zip(all_names, all_values))

tag_names   = features_meta['tags']
image_names = features_meta['image_feature_names']

tag_values   = [feat_dict[n] for n in tag_names]   if tag_names   else []
image_values = [feat_dict[n] for n in image_names] if image_names else []

n_panels = (1 if tag_names else 0) + (1 if image_names else 0)
fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, max(4, max(len(tag_names), len(image_names)) * 0.35)))
if n_panels == 1:
    axes = [axes]

panel = 0
if tag_names:
    ax = axes[panel]
    colors = ['steelblue' if v > 0 else 'lightgrey' for v in tag_values]
    ax.barh(tag_names, tag_values, color=colors)
    ax.set_title('Visual tag features\n(scaler-transformed boolean)')
    ax.set_xlabel('Value')
    ax.axvline(0, color='black', linewidth=0.8)
    panel += 1

if image_names:
    ax = axes[panel]
    ax.barh(image_names, image_values, color='coral')
    ax.set_title('GLCM + statistical features\n(scaler-transformed continuous)')
    ax.set_xlabel('Value')
    ax.axvline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.show()

## 8. Run model prediction

In [ ]:
resp = requests.post(
    f"{INFERENCE_URL}/predict/image",
    files={"image": ("image.jpg", _image_bytes, "image/jpeg")},
)
resp.raise_for_status()
pred = resp.json()

print(f"Score : {pred['score']:.4f}")
print(f"Label : {pred['label']}  ({'positive' if pred['label'] == 1 else 'negative'} class)")